In [1]:
"""
Agent Comparison — FSDS Cleaning Environment
=============================================
Benchmarks four agents on the held-out evaluation set and prints a
side-by-side comparison table.

Agents evaluated
----------------
  1. RandomAgent    — lower bound (uniform random actions)
  2. HeuristicAgent — upper bound (scripted oracle policy)
  3. SFT model      — supervised fine-tuned checkpoint warm-start
  4. GRPO model     — RL-trained checkpoint (SFT → GRPO)

Run in Colab after both training files have completed:
  - training_sft.py  → ./data-cleaning-sft-final
  - training_colab.py → ./data-cleaning-grpo-final
"""


!pip uninstall -y vllm                   # remove any existing vllm build
!pip install -q "openenv-core[core]>=0.2.1"
!pip install -q git+https://github.com/meta-pytorch/OpenEnv.git
!pip install -q "trl>=0.12.0" "accelerate>=0.34.0" "peft>=0.13.0" "bitsandbytes>=0.43.0" "datasets>=2.20.0" "protobuf>=3.20.3,<5.0.0"
!pip install -q unsloth                  # pins transformers; must be last
!pip install -q "git+https://huggingface.co/spaces/israaaML/fsds_cleaning_env"

# Now in Colab, run Cell 1 then Runtime → Restart session before running any other cell. The uninstall must happen before Python loads
#   TRL, otherwise the already-imported broken vllm stays in memory.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.7/633.7 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.9/201.9 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 34.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 30.

In [ ]:
!unzip data-cleaning-grpo-final.zip
!unzip data-cleaning-sft-final.zip

In [2]:
# ── Cell 2 ▸ Imports & config ─────────────────────────────────────────
# %%
import json
from pathlib import Path

ENV_URL          = "https://israaaML-fsds-cleaning-env.hf.space"
SFT_MODEL_PATH   = "./data-cleaning-sft-final"
GRPO_MODEL_PATH  = "./data-cleaning-grpo-final"
EPISODES_PER_TASK = 3          # increase for more reliable estimates (slower)
OUTPUT_FILE      = "./results_comparison.json"


In [4]:
# ── Cell 3 ▸ Connect to environment & sanity check ────────────────────
# %%
from fsds_cleaning_env import FSDSCleaningEnv
from fsds_cleaning_env.evaluation_tasks import EVAL_TASKS
from fsds_cleaning_env.evaluate_agent import run_evaluation
from fsds_cleaning_env.agents import RandomAgent, HeuristicAgent, LLMAgent
from fsds_cleaning_env.metrics import aggregate_metrics, compute_episode_metrics

with FSDSCleaningEnv(base_url=ENV_URL).sync() as env:
    env.reset(task_id="ecommerce_mobile")
    brief = env.call_tool("get_task_brief")
    print(f"Connected to env. Task: {brief.get('title')}")
    tasks_list = env.call_tool("list_tasks")
    print(f"Available tasks: {[t['task_id'] for t in tasks_list.get('tasks', [])]}")
print(f"\nEval tasks ({len(EVAL_TASKS)} scenarios):")
for t in EVAL_TASKS:
    print(f"  {t.name}  (task_id={t.task_id}, seed_index={t.eval_index})")


Connected to env. Task: Mobile conversion cleaning


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Available tasks: ['ecommerce_mobile', 'subscription_churn', 'delivery_eta']

Eval tasks (15 scenarios):
  ecommerce_mobile_baseline_seed0  (task_id=ecommerce_mobile, seed_index=0)
  ecommerce_mobile_baseline_seed1  (task_id=ecommerce_mobile, seed_index=1)
  ecommerce_mobile_baseline_seed2  (task_id=ecommerce_mobile, seed_index=2)
  ecommerce_mobile_baseline_seed3  (task_id=ecommerce_mobile, seed_index=3)
  ecommerce_mobile_baseline_seed4  (task_id=ecommerce_mobile, seed_index=4)
  subscription_churn_baseline_seed0  (task_id=subscription_churn, seed_index=0)
  subscription_churn_baseline_seed1  (task_id=subscription_churn, seed_index=1)
  subscription_churn_baseline_seed2  (task_id=subscription_churn, seed_index=2)
  subscription_churn_baseline_seed3  (task_id=subscription_churn, seed_index=3)
  subscription_churn_baseline_seed4  (task_id=subscription_churn, seed_index=4)
  delivery_eta_baseline_seed0  (task_id=delivery_eta, seed_index=0)
  delivery_eta_baseline_seed1  (task_id=delivery

In [ ]:
# ── Cell 4 ▸ Run all agents ───────────────────────────────────────────
# %%
results = {}


# ── 4a. Random (lower bound) ──────────────────────────────────────────
print("\n[1/4] Evaluating RandomAgent …")
results["random"] = run_evaluation(
    RandomAgent(),
    base_url=ENV_URL,
    max_episodes_per_task=EPISODES_PER_TASK,
)
agg = results["random"]["aggregate"]
print(f"  success={agg['success_rate']:.0%}  return={agg['avg_return']:.4f}  steps={agg['avg_steps']:.1f}")

# ── 4b. Heuristic (upper bound) ───────────────────────────────────────
print("\n[2/4] Evaluating HeuristicAgent …")
results["heuristic"] = run_evaluation(
    HeuristicAgent(),
    base_url=ENV_URL,
    max_episodes_per_task=EPISODES_PER_TASK,
)
agg = results["heuristic"]["aggregate"]
print(f"  success={agg['success_rate']:.0%}  return={agg['avg_return']:.4f}  steps={agg['avg_steps']:.1f}")



In [5]:

# ── 4c. SFT model ─────────────────────────────────────────────────────
print(f"\n[3/4] Evaluating SFT model ({SFT_MODEL_PATH}) …")
results["sft"] = run_evaluation(
    LLMAgent(model_path=SFT_MODEL_PATH, temperature=0.0),
    base_url=ENV_URL,
    max_episodes_per_task=EPISODES_PER_TASK,
)
agg = results["sft"]["aggregate"]
print(f"  success={agg['success_rate']:.0%}  return={agg['avg_return']:.4f}  steps={agg['avg_steps']:.1f}")

# ── 4d. GRPO model ────────────────────────────────────────────────────
print(f"\n[4/4] Evaluating GRPO model ({GRPO_MODEL_PATH}) …")
results["grpo"] = run_evaluation(
    LLMAgent(model_path=GRPO_MODEL_PATH, temperature=0.0),
    base_url=ENV_URL,
    max_episodes_per_task=EPISODES_PER_TASK,
)
agg = results["grpo"]["aggregate"]
print(f"  success={agg['success_rate']:.0%}  return={agg['avg_return']:.4f}  steps={agg['avg_steps']:.1f}")




[1/4] Evaluating RandomAgent …
  success=0%  return=-0.1174  steps=4.0

[2/4] Evaluating HeuristicAgent …
  success=100%  return=3.1038  steps=17.7

[3/4] Evaluating SFT model (./data-cleaning-sft-final) …
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:362: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a

🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

unsloth/qwen2.5-1.5b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File

  success=0%  return=0.0000  steps=13.6

[4/4] Evaluating GRPO model (./data-cleaning-grpo-final) …


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ConnectionError: Failed to connect to wss://israaaML-fsds-cleaning-env.hf.space/ws: server rejected WebSocket connection: HTTP 503

In [6]:
# ── Cell 5 ▸ Comparison table ─────────────────────────────────────────
# %%
AGENTS = [
    ("Random",    "random"),
    ("Heuristic", "heuristic"),
    ("SFT",       "sft"),
    ("GRPO",      "grpo"),
]

COL_W = 12

def _col(v, w=COL_W):
    return str(v).center(w)

header = (
    f"{'Agent':<14}"
    + _col("Success %")
    + _col("Avg Return")
    + _col("Avg Steps")
    + _col("Avg Invalid")
    + _col("Episodes")
)
sep = "-" * len(header)

print("\n" + sep)
print("  FSDS Cleaning Agent Benchmark")
print(sep)
print(header)
print(sep)

for label, key in AGENTS:
    if key not in results:
        continue
    agg = results[key]["aggregate"]
    print(
        f"{label:<14}"
        + _col(f"{agg['success_rate']:.1%}")
        + _col(f"{agg['avg_return']:.4f}")
        + _col(f"{agg['avg_steps']:.1f}")
        + _col(f"{agg['avg_invalid_actions']:.2f}")
        + _col(agg["episodes"])
    )

print(sep)

# Improvement of GRPO over SFT
if "sft" in results and "grpo" in results:
    sft_sr  = results["sft"]["aggregate"]["success_rate"]
    grpo_sr = results["grpo"]["aggregate"]["success_rate"]
    sft_ret  = results["sft"]["aggregate"]["avg_return"]
    grpo_ret = results["grpo"]["aggregate"]["avg_return"]
    print(f"\nGRPO vs SFT — success rate delta : {grpo_sr - sft_sr:+.1%}")
    print(f"GRPO vs SFT — avg return delta   : {grpo_ret - sft_ret:+.4f}")



--------------------------------------------------------------------------
  FSDS Cleaning Agent Benchmark
--------------------------------------------------------------------------
Agent          Success %   Avg Return  Avg Steps  Avg Invalid   Episodes  
--------------------------------------------------------------------------
Random            0.0%      -0.1174       4.0         0.00         45     
Heuristic        100.0%      3.1038       17.7        0.00         45     
SFT               0.0%       0.0000       13.6        0.00         45     
--------------------------------------------------------------------------


In [ ]:
# ── Cell 6 ▸ Per-task breakdown ───────────────────────────────────────
# %%
# Group per-episode results by task_id for a fine-grained breakdown.
from collections import defaultdict

print("\n=== Per-task success rates ===")
task_ids = sorted({ep["task_id"] for ep in results["heuristic"]["episodes"]})

col_labels = [label for label, _ in AGENTS if _ in results]
keys       = [key   for _, key in AGENTS if key in results]

# Header
print(f"\n{'Task':<30}" + "".join(f"{lbl:>12}" for lbl in col_labels))
print("-" * (30 + 12 * len(col_labels)))

for tid in task_ids:
    row = f"{tid:<30}"
    for key in keys:
        eps = [e for e in results[key]["episodes"] if e["task_id"] == tid]
        if not eps:
            row += f"{'N/A':>12}"
        else:
            sr = sum(1 for e in eps if e.get("success", False)) / len(eps)
            row += f"{sr:>11.0%} "
    print(row)


In [ ]:
# ── Cell 7 ▸ Save results ─────────────────────────────────────────────
# %%
Path(OUTPUT_FILE).write_text(json.dumps(results, indent=2))
print(f"\nFull results saved to {OUTPUT_FILE}")
